In [ ]:
import shutil

# Define source and destination paths
source_path = "/content/drive/MyDrive/DataSet/DataSEt.zip"
destination_path = "/content/"  # Temp location

# Copy the file
shutil.copy(source_path, destination_path)

print("File copied successfully!")

File copied successfully!


In [ ]:
# install 7‑zip
!apt-get update -qq && apt-get install -y p7zip-full -qq

# extract with all cores (the -y auto‑answers prompts)
!7z x /content/DataSEt.zip -o/content/extracted_files/DataSEt -y


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,2 CPUs Intel(R) Xeon(R) CPU @ 2.00GHz (50653),ASM,AES-NI)

Scanning the drive for archives:
  0M Scan /content/                   1 file, 16605888739 bytes (16 GiB)

Extracting archive: /content/DataSEt.zip
--
Path = /content/DataSEt.zip
Type = zip
Physical Size = 16605888739
64-bit = +

  0%      0% 68 - DataSEt/s09_060317n.set/s09_060317n.fdt                                                   0% 69         0% 70 - DataSEt/s41_091104n.set/s41_091104n.fdt       

In [ ]:
# 0. prerequisites
!pip install mne --quiet
import os, gc
import mne

base_dir   = '/content/extracted_files/DataSEt'
fif_dir    = '/content/extracted_files/fif'  # where we'll store the per‐subject FIFs
os.makedirs(fif_dir, exist_ok=True)

# 1. find all your .set files
set_paths = []
for root, _, files in os.walk(base_dir):
    for fname in files:
        if fname.endswith('.set'):
            set_paths.append(os.path.join(root, fname))
print(f"→ {len(set_paths)} .set files found.")

# 2. one‐by‐one: read with preload=False, save to FIF (.dat gets memory‐mapped)
fif_paths = []
for path in set_paths:
    raw = mne.io.read_raw_eeglab(path, preload=False, verbose=False)
    out_fname = os.path.join(
        fif_dir,
        os.path.basename(path).replace('.set', '-raw.fif')
    )
    raw.save(out_fname, overwrite=True)      # writes out .fif + .fif.dat (mmap)
    fif_paths.append(out_fname)
    # free Python memory before next loop
    del raw
    gc.collect()
print("→ All subjects converted to memory‐mapped FIFs.")

# 3. load all those FIFs with preload=False


→ 62 .set files found.
Writing /content/extracted_files/fif/s55_090930n-raw.fif
Closing /content/extracted_files/fif/s55_090930n-raw.fif
[done]
Writing /content/extracted_files/fif/s04_051130m-raw.fif
Closing /content/extracted_files/fif/s04_051130m-raw.fif
[done]
Writing /content/extracted_files/fif/s42_070105n-raw.fif
Closing /content/extracted_files/fif/s42_070105n-raw.fif
[done]
Writing /content/extracted_files/fif/s44_070209m-raw.fif
Closing /content/extracted_files/fif/s44_070209m-raw.fif
[done]
Writing /content/extracted_files/fif/s22_080513m-raw.fif
Closing /content/extracted_files/fif/s22_080513m-raw.fif
[done]
Writing /content/extracted_files/fif/s49_080522n-raw.fif
Closing /content/extracted_files/fif/s49_080522n-raw.fif
[done]
Writing /content/extracted_files/fif/s05_061101n-raw.fif
Closing /content/extracted_files/fif/s05_061101n-raw.fif
[done]
Writing /content/extracted_files/fif/s22_090922m-raw.fif
Closing /content/extracted_files/fif/s22_090922m-raw.fif
[done]
Writing /

In [ ]:
!zip -r merged-raw.zip merged-raw.fif merged-raw.fif.dat


	zip warning: name not matched: merged-raw.fif.dat
  adding: merged-raw.fif (deflated 7%)


In [ ]:
import os
import mne
import gc

base_dir = '/content/extracted_files/DataSEt'

# Step 1: Find all .set files
set_paths = []
for root, _, files in os.walk(base_dir):
    for fname in files:
        if fname.endswith('.set'):
            set_paths.append(os.path.join(root, fname))
print(f"→ {len(set_paths)} .set files found.")

# Step 2: Load and append Raw objects from .set files (use preload=False to save RAM)
raw_list = []
for path in set_paths:
    raw = mne.io.read_raw_eeglab(path, preload=False, verbose=False)
    raw_list.append(raw)
    print(f"Loaded {os.path.basename(path)} with {len(raw.ch_names)} channels, {raw.n_times} samples")
    gc.collect()

# Step 3: Merge into one Raw object
merged_raw = mne.concatenate_raws(raw_list)
print(f"\n✅ Merged object ready with {len(merged_raw.ch_names)} channels and {merged_raw.n_times} total samples.")


→ 62 .set files found.
Loaded s55_090930n.set with 30 channels, 2705920 samples
Loaded s04_051130m.set with 30 channels, 1878660 samples
Loaded s42_070105n.set with 30 channels, 3257440 samples
Loaded s44_070209m.set with 30 channels, 3407680 samples
Loaded s22_080513m.set with 30 channels, 1950080 samples
Loaded s49_080522n.set with 30 channels, 1511060 samples
Loaded s05_061101n.set with 30 channels, 3296100 samples
Loaded s22_090922m.set with 30 channels, 2749540 samples
Loaded s12_060710_1m.set with 30 channels, 2117620 samples
Loaded s41_080520m.set with 30 channels, 1910620 samples
Loaded s45_070307n.set with 30 channels, 3267560 samples
Loaded s42_061229n.set with 30 channels, 1344040 samples
Loaded s31_061020m.set with 30 channels, 3530520 samples
Loaded s44_070205n.set with 30 channels, 3199960 samples
Loaded s41_061225n.set with 30 channels, 3176660 samples
Loaded s41_080530n.set with 30 channels, 1887040 samples
Loaded s44_070325n.set with 30 channels, 3102200 samples
Loaded

In [ ]:
out_path = '/content/merged_raw.fif'
merged_raw.save(out_path, overwrite=True)
print(f"✅ Merged Raw saved to {out_path}")


Writing /content/merged_raw.fif
Closing /content/merged_raw.fif
Writing /content/merged_raw-1.fif
Closing /content/merged_raw-1.fif
Writing /content/merged_raw-2.fif
Closing /content/merged_raw-2.fif
Writing /content/merged_raw-3.fif
Closing /content/merged_raw-3.fif
Writing /content/merged_raw-4.fif
Closing /content/merged_raw-4.fif
Writing /content/merged_raw-5.fif
Closing /content/merged_raw-5.fif
Writing /content/merged_raw-6.fif
Closing /content/merged_raw-6.fif
Writing /content/merged_raw-7.fif
Closing /content/merged_raw-7.fif
Writing /content/merged_raw-8.fif
Closing /content/merged_raw-8.fif
[done]
✅ Merged Raw saved to /content/merged_raw.fif


In [ ]:
# Use tar to create a compressed archive (faster than zip for large files)
!tar -czvf merged_raw_all.tar.gz /content/merged_raw*.fif


tar: Removing leading `/' from member names
/content/merged_raw-1.fif
tar: Removing leading `/' from hard link targets
/content/merged_raw-2.fif
/content/merged_raw-3.fif
/content/merged_raw-4.fif
/content/merged_raw-5.fif
/content/merged_raw-6.fif
/content/merged_raw-7.fif
/content/merged_raw-8.fif
/content/merged_raw.fif


In [ ]:
from google.colab import files
files.download('merged_raw_all.tar.gz')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!du -sh /content/merged_raw*


2.0G	/content/merged_raw-1.fif
2.0G	/content/merged_raw-2.fif
2.0G	/content/merged_raw-3.fif
2.0G	/content/merged_raw-4.fif
2.0G	/content/merged_raw-5.fif
2.0G	/content/merged_raw-6.fif
2.0G	/content/merged_raw-7.fif
521M	/content/merged_raw-8.fif
16G	/content/merged_raw_all.tar.gz
2.0G	/content/merged_raw.fif


In [ ]:
!mkdir -p /content/drive/MyDrive/EEG_Merged


In [ ]:
!cp /content/merged_raw*.fif /content/drive/MyDrive/EEG_Merged/


In [ ]:
import mne
import glob
raw = mne.io.read_raw_fif('/content/merged_raw.fif', preload=False, verbose=False)

print(f"\n✅ Merged Raw object has {len(raw.ch_names)} channels and {raw.n_times} total samples.")



✅ Merged Raw object has 30 channels and 147529260 total samples.
